In [5]:
import re
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType
from pyspark.sql.functions import explode, lower, col, collect_set, trim, regexp_replace
from pyspark.sql import functions as F

# In cluster use the HDFS path prefix
path_prefix = "hdfs:///projects/BDA-12/"
# In local use the local path prefix
# path_prefix = "" 

# --- CONFIGURABLE PARAMETERS ---
# Only one of these should be True at a time!
profile = 'healthy'  # Options: 'none', 'healthy', 'high_protein', 'high_fiber'
ingredients = ['banana', 'milk', 'sugar']  # e.g. ['banana', 'milk', 'sugar', 'potato', 'almonds']
kcal_min = None   # e.g. 100
kcal_max = None   # e.g. 300
dislikes = []     # e.g. ['onion', 'garlic']
top_n = 10        # Number of recipe ideas to return

In [6]:
# --- LOAD DATA ---
spark = SparkSession.builder.appName('RecipeIdeas').getOrCreate()
# Ingredients
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
])
df_ing = spark.read.schema(ingredients_schema).parquet(
    f'{path_prefix}/output/ingredients_nutrional_profiles/'
)
# Nutrients
df_nutri = spark.read.parquet(f'{path_prefix}/output/nutritional_profiles')
# Merge (keep nutrient columns from df_nutri)
df = df_ing.join(df_nutri, df_ing.fdc_id == df_nutri.fdc_id, 'inner')
cols = [df_ing.fdc_id, df_ing.description, df_ing.all_ingredients] + [c for c in df_nutri.columns if c != 'fdc_id']
df = df.select(*cols)
# Work with Spark DataFrame only; name it `foods_df`
foods_df = df

In [7]:
# --- INGREDIENT NORMALIZATION (PySpark) ---
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.functions import udf, col
# Pattern mirrors the original normalization logic
_pattern = re.compile(r'[()\"\{\}\.,:;\-&]')
def _normalize_list(lst):
    if lst is None:
        return []
    out = []
    for x in lst:
        if x is None:
            continue
        s = _pattern.sub('', x.lower().strip())
        if s:
            out.append(s)
    # dedupe while preserving order
    seen = set()
    res = []
    for v in out:
        if v not in seen:
            seen.add(v)
            res.append(v)
    return res
normalize_udf = udf(_normalize_list, ArrayType(StringType()))
# Add `ingredients_set` column as an array of normalized, distinct ingredient strings
foods_df = foods_df.withColumn('ingredients_set', normalize_udf(col('all_ingredients')))

In [8]:
# --- FILTERING FUNCTION (PySpark) ---
from pyspark.sql.functions import array, lit, size, col
def _normalize_string(s):
    if s is None:
        return ''
    return _pattern.sub('', s.lower().strip())
def filter_foods_df(df, ingredients, dislikes, kcal_min, kcal_max):
    res = df
    # Filter by dislikes: remove rows where any dislike appears in ingredients_set
    if dislikes:
        dislikes_norm = [_normalize_string(d) for d in dislikes]
        if dislikes_norm:
            dislikes_array = array(*[lit(d) for d in dislikes_norm])
            res = res.filter(size(F.array_intersect(col('ingredients_set'), dislikes_array)) == 0)
    # Filter by kcal range
    if kcal_min is not None:
        res = res.filter(col('energy').isNotNull() & (col('energy') >= kcal_min))
    if kcal_max is not None:
        res = res.filter(col('energy').isNotNull() & (col('energy') <= kcal_max))
    # Filter by available ingredients: at least min(3, len(ingredients)) matches
    if ingredients:
        ing_norm = [_normalize_string(i) for i in ingredients]
        if ing_norm:
            min_match = min(3, len(ing_norm))
            ing_array = array(*[lit(i) for i in ing_norm])
            res = res.filter(size(F.array_intersect(col('ingredients_set'), ing_array)) >= min_match)
    return res

In [9]:
# --- SCORING FUNCTION (PySpark expression) ---
# Default weights for each profile
PROFILE_WEIGHTS = {
    'none':     {'fiber': 0.0,   'protein': 0.0,   'sugars': 0.0,   'total_fat': 0.0,   'energy': 0.0},
    'healthy':  {'fiber': 1.5, 'protein': 1.2, 'sugars': -1.5, 'total_fat': -0.8, 'energy': -0.01},
    'high_protein': {'fiber': 0.0, 'protein': 2.0, 'sugars': 0.0, 'total_fat': 0.0, 'energy': 0.0},
    'high_fiber':   {'fiber': 2.0, 'protein': 0.0, 'sugars': 0.0, 'total_fat': 0.0, 'energy': 0.0}
}

def compute_score_expr(profile, ing_list):
    # Build Spark expression for the recipe score
    weights = PROFILE_WEIGHTS.get(profile, PROFILE_WEIGHTS['none'])
    # Ingredient matches count
    ing_array = F.array(*[F.lit(i) for i in ing_list]) if ing_list else F.array()
    matches = F.size(F.array_intersect(col('ingredients_set'), ing_array))
    score_expr = F.lit(2.0) * matches
    # Add nutrient-weighted contributions (coalesce to 0 for nulls)
    score_expr = score_expr + F.lit(weights['fiber']) * F.coalesce(col('fiber'), F.lit(0.0))
    score_expr = score_expr + F.lit(weights['protein']) * F.coalesce(col('protein'), F.lit(0.0))
    score_expr = score_expr + F.lit(weights['sugars']) * F.coalesce(col('sugars'), F.lit(0.0))
    score_expr = score_expr + F.lit(weights['total_fat']) * F.coalesce(col('total_fat'), F.lit(0.0))
    score_expr = score_expr + F.lit(weights['energy']) * F.coalesce(col('energy'), F.lit(0.0))
    return score_expr

In [10]:
# --- MAIN RECIPE SUGGESTION LOGIC (PySpark) ---
assert profile in ['none', 'healthy', 'high_protein', 'high_fiber'], "Profile must be one of: 'none', 'healthy', 'high_protein', 'high_fiber'"
# Normalize input ingredients to match the normalization UDF logic
ing_norm = [_normalize_string(i) for i in ingredients] if ingredients else []
# Apply filtering using Spark
filtered_df = filter_foods_df(foods_df, ingredients, dislikes, kcal_min, kcal_max)
# Check if any results
if len(filtered_df.take(1)) == 0:
    print('No recipe ideas found for your configuration.')
else:
    # Build score expression and add column
    score_expr = compute_score_expr(profile, ing_norm)
    scored = filtered_df.withColumn('recipe_score', score_expr)
    scored = scored.orderBy(F.desc('recipe_score'))
    print('Top recipe ideas:')
    # Show results using Spark's native display
    scored.select('description', 'ingredients_set', 'energy', 'fiber', 'protein', 'sugars', 'total_fat', 'recipe_score').show(top_n, False)

26/01/15 18:28:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Top recipe ideas:


+-------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-----+-------+------+---------+------------+
|description                                      |ingredients_set                                                                                                                                                                                                                                                                                                                                                                                         |energy|fiber|protein|sugars|total_fat|recipe_score|
+-------

---
**How it works:**
- Enter your available ingredients, dislikes, kcal range, and nutrition preferences at the top.
- The notebook will suggest foods/recipes that match your configuration, prioritizing ingredient overlap and nutrition profile.
- You can easily adjust the scoring logic for your needs.